In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('./cleaned_data.csv')

In [5]:
df.columns

Index(['duration', 'orig_bytes', 'resp_bytes', 'missed_bytes', 'orig_pkts',
       'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes', 'label', 'proto_icmp',
       'proto_tcp', 'proto_udp', 'conn_state_OTH', 'conn_state_REJ',
       'conn_state_RSTO', 'conn_state_RSTOS0', 'conn_state_RSTR',
       'conn_state_RSTRH', 'conn_state_S0', 'conn_state_S1', 'conn_state_S2',
       'conn_state_S3', 'conn_state_SF', 'conn_state_SH', 'conn_state_SHR'],
      dtype='object')

In [10]:
df['label'].value_counts()

label
Benign                       26001
PartOfAHorizontalPortScan    12369
C&C                           5618
Attack                        3814
Okiru                          163
DDoS                            36
FileDownload                     2
Name: count, dtype: int64

In [4]:
# Map labels to binary classes
binary_labels = {
    'Benign': 0,
    'PartOfAHorizontalPortScan': 1,
    'C&C': 2,
    'Attack': 3,
    'Okiru': 4,
    'DDoS': 5,
    'FileDownload': 6
}

# Apply binary labeling
df['binary_label'] = df['label'].map(binary_labels)

In [5]:
df['binary_label'].value_counts()

binary_label
0    26001
1    12369
2     5618
3     3814
4      163
5       36
6        2
Name: count, dtype: int64

In [6]:
# Splitting the dataset into features and labels
X = df.drop(['label', 'binary_label'], axis=1)  # Features
y = df['binary_label']                        # Binary target

In [7]:
from sklearn.model_selection import train_test_split
# Split into training and test sets (70-30 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
from imblearn.combine import SMOTEENN

# Apply SMOTEENN
smote_enn = SMOTEENN(sampling_strategy='auto', random_state=42)
X_resampled, y_resampled = smote_enn.fit_resample(X_train, y_train)

print(f"Original Training Set Size: {X_train.shape[0]}")
print(f"Resampled Training Set Size: {X_resampled.shape[0]}")

ValueError: Expected n_neighbors <= n_samples_fit, but n_neighbors = 6, n_samples_fit = 2, n_samples = 2

In [13]:


# Flow dataset
flow_features = ['duration', 'orig_bytes', 'resp_bytes', 'missed_bytes',
                 'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes']
flow_df = df[flow_features]

# Flag dataset
flag_features = ['conn_state_OTH', 'conn_state_REJ', 'conn_state_RSTO', 'conn_state_RSTOS0',
                 'conn_state_RSTR', 'conn_state_RSTRH', 'conn_state_S0', 'conn_state_S1',
                 'conn_state_S2', 'conn_state_S3', 'conn_state_SF', 'conn_state_SH', 'conn_state_SHR']
flag_df = df[flag_features]

# Packet dataset (protocol indicators as proxy for packets)
packet_features = ['proto_icmp', 'proto_tcp', 'proto_udp']
packet_df = df[packet_features]

# ==========================
# ✅ Save the Split Datasets
# ==========================
flow_df.to_csv("flow_dataset.csv", index=False)
flag_df.to_csv("flag_dataset.csv", index=False)
packet_df.to_csv("packet_dataset.csv", index=False)

print("✅ Datasets saved successfully!")
print(f"Flow Dataset Shape: {flow_df.shape}")
print(f"Flag Dataset Shape: {flag_df.shape}")
print(f"Packet Dataset Shape: {packet_df.shape}")

✅ Datasets saved successfully!
Flow Dataset Shape: (48003, 8)
Flag Dataset Shape: (48003, 13)
Packet Dataset Shape: (48003, 3)
